# 06. Operational Space and Data-Driven Plasma Analysis

Sessions 01–05 went deeper into one discharge at a time. This session turns the other way: many
discharges and many reconstructed equilibria become a **population of plasma states**, and the
question becomes where VEST has operated and what is known about the MHD stability of the states
it reached.

```text
Session 01   load and understand a discharge
Session 02   operate and form the plasma
Session 03   reconstruct the plasma state
Session 04   measure perturbations and events
Session 05   model stability and 3-D response
      ↓
Session 06   place many plasma states in operational and stability space      <- here
```

The session follows the same order as the analysis itself:

$$
\underbrace{\text{database} \rightarrow \text{summary row}}_{\text{A: computation}}
\;\rightarrow\;
\underbrace{\text{equilibrium} \rightarrow \beta,\ l_i,\ q,\ I_N}_{\text{B: physics}}
\;\rightarrow\;
\underbrace{\text{stability limits and sources}}_{\text{C: references}}
\;\rightarrow\;
\underbrace{\text{summary tables}}_{\text{D: data}}
\;\rightarrow\;
\underbrace{\text{operating space and stability}}_{\text{E: synthesis}}
$$

## Session Overview

**Question.** Where in operational and MHD-stability space do the reconstructed VEST plasma
states lie, and which statements about limits does the evidence actually support?

**After this session you can**

1. follow a summary row from the database: lazy versus eager access, the preset contract, and the
   physics functions an extractor calls to fill each column (**Part A**);
2. derive the global MHD parameters — $W_\mathrm{MHD}$, $\beta_t$, $\beta_p$, $\beta_N$, $l_i(3)$, $q^*$,
   $I_N$ — from their integral definitions in `vaft.formula`, and reproduce the summary values (**Part B**);
3. name the empirical and model stability limits used on operational-space planes, their source,
   their quantity axes and what they may be used for, with their `vaft.diagram` schematics (**Part C**);
4. read the summary tables — archived history, the three presets, the validated Tier A
   population — with their grain, coverage and provenance (**Part D**);
5. place that population on the operational-space planes with the registered limits, join the
   stability atlas by state key and time, and keep ideal, resistive and local verdicts and their QA
   separate (**Part E**).

**Data.** Everything below runs offline from repository-only sample collections read through
`vaft.data.data_path`:

| collection | what | grain |
|---|---|---|
| `samples/vest_operational_history/` | archived per-shot history, 2257 shots 38000–45767 | shot |
| `samples/vest_tier_a/v1/` | Tier A equilibrium states (EFIT `statistical_891`, good/admissible) | state |
| `samples/vest_tier_a/confinement/` | geometry, $\beta_N$, $l_i$, Ohmic power balance of the magnetics states | state |
| `samples/vest_tier_a/stability/` | stability atlas v2: DCON $n=1..6$, RDCON/STRIDE $n=1,2$, Mercier/ballooning | state × $n$ (× surface) |
| `vaft.omas.sample_ods(...)` | five packaged sample shots, to run the summary extractors themselves | shot / slice |

Direct database extraction is optional: `VAFT_TUTORIAL_MODE=lab` with HSDS read access, read-only.

## Physical Context

### 1. Four different things are often drawn on one plane

| term | meaning | evidence it needs |
|---|---|---|
| **observed envelope** | the region the recorded states happen to fill | the population itself |
| **accessible operating space** | what the machine *could* reach with its coils, power and fuelling | engineering limits, scans |
| **empirical / theoretical boundary** | a line from a survey or a model (Troyon, Greenwald, kink $q^*$, Wesson $l_i$–$q$) | its source, its domain, its quantity definitions |
| **solver classification** | a stability code's verdict on one reconstructed equilibrium | the solver, its inputs, its numerical QA |

An empty region is not by itself an operating limit, and a boundary is a reference drawn for
comparison, not a VEST limit, until its domain is shown to include VEST.

### 2. The safety factor has several identities

$q_{95}$ (flux-surface $q$ at $\psi_N=0.95$), $q_\psi$ at the last closed flux surface, the
cylindrical $q(a)$ or $q_\mathrm{cyl}$, and the kink parameter $q^*$ are **different quantities**.
A boundary drawn in one of them cannot receive points measured in another without a stated physical
transformation. The VAFT projection registry (`vaft.diagram`, `vaft.formula.boundaries`) attaches a
quantity identity and a unit to every axis, and `vaft.plot.operational_space` refuses to draw a
boundary whose axis does not match the column it is given.

### 3. Normalisations hide field conventions

$I_N = I_p/(aB_T)$ and $\beta_N = \beta_T a B_T/I_p$ depend on *which* $B_T$. The coordinate helpers
in `vaft.formula.boundaries` use the vacuum field at the geometric centre $R_\mathrm{geo}$; VAFT's
stored $\beta_N$ uses the vacuum field at the reference radius $R_0 = 0.4\ \mathrm{m}$. Because the
vacuum field scales as $1/R$, $\beta_N$ at $R_\mathrm{geo}$ is $\beta_N(R_0)\,R_\mathrm{geo}/R_0$.
This session converts once, says so, and prints the factor.

### 4. Stability evidence comes in layers

1. **raw solver output** — DCON's total energy $W_t$ per toroidal mode number $n$ and edge treatment;
   RDCON/STRIDE's $\Delta'$ at each rational surface;
2. **numerical QA** — does the sign survive a resolution change (`sign_class`,
   `two_resolution_consistent`)?
3. **physical criterion** — ideal: $W_t<0$; classical tearing drive: $\Delta'>0$ at a rational
   surface; Mercier $D_I>0$; resistive interchange $D_R>0$; ballooning $C_A<0$;
4. **interpretation** — not produced here. $\Delta'$ is an outer-region quantity: a positive value is a
   classical drive, not a tearing verdict, and none of these numbers is a growth rate.

Layers are never collapsed: a state can be ideal-stable at every $n$, carry a QA-supported
$\Delta'>0$, and be Mercier-stable at once. A calculation that failed or did not converge is
**not assessed**, never stable.

## Load / Prepare Data

### The collections, their manifests and their native grains

Every table is read through `vaft.data.data_path`. The manifest records where each file came from;
the cell checks the SHA-256 of every file used against it before any number is computed.

In [ ]:
import hashlib
import json
import logging
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import vaft
from vaft.data import data_path, public
from vaft.formula import boundaries as fb
from vaft.plot.operational_space import li_qa_pair, operational_space_population
from vaft.plot.stability_atlas import stability_atlas_population
from vaft.process import mhd_stability

logging.getLogger("vaft").setLevel(logging.ERROR)
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)  # wide atlas tables gain key columns
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})

MODE = os.environ.get("VAFT_TUTORIAL_MODE", "offline")
LAB = MODE == "lab"
# default: tutorial/outputs/06 (gitignored), whether the kernel starts in the repository root or in tutorial/
OUTPUT_DIR = Path(os.environ.get("VAFT_TUTORIAL_OUTPUT_DIR",
                                 Path("tutorial/outputs/06") if Path("tutorial").is_dir() else Path("outputs/06")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"mode: {MODE}")

In [ ]:
TIER_A = data_path("samples/vest_tier_a")
HISTORY = data_path("samples/vest_operational_history")
tier_a_manifest = json.loads((TIER_A / "manifest.json").read_text())
history_manifest = json.loads((HISTORY / "manifest.json").read_text())

USED = {
    "history": (HISTORY, "omas_history_new.xlsx", history_manifest["files"]["omas_history_new.xlsx"]["sha256"]),
    "state": (TIER_A, "v1/state.csv", tier_a_manifest["sha256"]["v1/state.csv"]),
    "confinement": (TIER_A, "confinement/table.csv", tier_a_manifest["sha256"]["confinement/table.csv"]),
    "atlas_n": (TIER_A, "stability/atlas_n.csv.gz", tier_a_manifest["sha256"]["stability/atlas_n.csv.gz"]),
    "atlas_surfaces": (TIER_A, "stability/atlas_surfaces.csv.gz", tier_a_manifest["sha256"]["stability/atlas_surfaces.csv.gz"]),
}
rows = []
for name, (root, relative, expected) in USED.items():
    actual = hashlib.sha256((root / relative).read_bytes()).hexdigest()
    rows.append({"table": name, "file": f"{root.name}/{relative}", "sha256": actual[:12], "matches manifest": actual == expected})
checks = pd.DataFrame(rows)
display(checks)
if not checks["matches manifest"].all():
    raise ValueError("a sample file differs from its manifest; do not interpret the numbers below")

print("Tier A:", tier_a_manifest["description"])
print("history limitations:")
for item in history_manifest["limitations"]:
    print("  -", item)

In [ ]:
history = pd.read_excel(HISTORY / "omas_history_new.xlsx")
state = pd.read_csv(TIER_A / "v1/state.csv")
confinement = pd.read_csv(TIER_A / "confinement/table.csv")
atlas_n = pd.read_csv(TIER_A / "stability/atlas_n.csv.gz", low_memory=False)
atlas_surfaces = pd.read_csv(TIER_A / "stability/atlas_surfaces.csv.gz", low_memory=False)

# One state key for every table: shot, EFIT time in integer microseconds, EFIT lineage.
# The stability atlas spells the lineages differently; map them explicitly and refuse anything unknown.
LINEAGE = {"magnetics": "magnetics", "electron_kinetic": "electron_kinetic",
           "magnetics-only": "magnetics", "electron-kinetic": "electron_kinetic"}
KEY = ["shot", "time_us", "efit_lineage"]
for table in (state, confinement, atlas_n, atlas_surfaces):
    table["time_us"] = (table["time_efit_s"] * 1e6).round().astype("int64")
    table["efit_lineage"] = table["efit_lineage"].map(LINEAGE)
    if table["efit_lineage"].isna().any():
        raise ValueError("unknown EFIT lineage label")

GRAIN = {
    "history": (history, ["shot"]),
    "state": (state, KEY),
    "confinement": (confinement, KEY),
    "atlas_n": (atlas_n, KEY + ["n_tor"]),
    "atlas_surfaces": (atlas_surfaces, KEY + ["n_tor", "solver", "m"]),
}
grain = pd.DataFrame([
    {"table": name, "rows": len(frame), "native grain": " x ".join(columns),
     "duplicate keys": int(frame.duplicated(columns).sum()),
     "states": len(frame[KEY].drop_duplicates()) if "time_us" in frame else None,
     "shots": frame["shot"].nunique()}
    for name, (frame, columns) in GRAIN.items()
])
display(grain)
if grain["duplicate keys"].any():
    raise ValueError("a table is not unique at its own grain")
print("state lineages:", state["efit_lineage"].value_counts().to_dict())

Five tables, five grains: one row per **shot** (history), per **state** (state, confinement), per
state and **toroidal mode number** (`atlas_n`), and per state, $n$, **solver** and **rational surface**
(`atlas_surfaces`). Each is unique at its own grain. The confinement table covers only the 133
magnetics-only states; the state table also holds 17 electron-kinetic reconstructions at the same
times. Joining a per-state table to `atlas_n` without first reducing it to one row per state
would multiply every state by six — the join report in Part E checks this explicitly.

## Guided Analysis

### Part A — Computational workflow: from the database to a summary row

A population table is not read from the database; it is **computed** from it, one shot at a time.
Each shot's ODS is opened (lazily, only the IDS the preset needs), an extractor turns it into one or
more rows, and the rows of all shots are assembled under one key. The VEST data platform puts the
per-shot database at the centre of that process.

In [ ]:
vaft.diagram.vest_data_platform()

#### A.1 — Lazy and eager access

| call | returns | use it for |
|---|---|---|
| `vaft.database.summary(odss=...)` | a DataFrame from ODS already in memory | offline teaching, tests, a reprocessed product before upload |
| `vaft.database.summary(shot_range=..., preset=...)` | a DataFrame over the database | campaign tables; opens each shot lazily with only the preset's IDS |
| `vaft.database.open(shot, paths=...)` | a **read-only, lazy** ODS, used as a context manager | reading a few leaves of many shots; nothing is fetched until accessed |
| `vaft.database.load(shot, ...)` | an **eager**, materialised ODS (or native IMAS) | modifying, saving locally, or running codes on a full shot |

Each call takes a `source` (a namespace such as `main`, the current pipeline, or the read-only
legacy `public`); the same shot number in two sources is two different reconstructions, which is why
every population above carries an `efit_lineage`. A lazy ODS is valid only inside its `with` block.

In [ ]:
SHOT = 41672
if LAB:
    with vaft.database.open(SHOT, paths=["equilibrium"]) as lazy:
        times = np.asarray(lazy["equilibrium.time"])
        print(f"lazy  : {SHOT} has {times.size} equilibrium slices; only the equilibrium IDS was requested")
    eager = vaft.database.load(SHOT)
    print(f"eager : {SHOT} materialised IDS: {sorted(eager.keys())}")
else:
    print("Offline: the database calls need lab mode (VAFT_TUTORIAL_MODE=lab and HSDS read access).\n"
          "Offline, the same extractors run on ODS already in memory: summary(odss=...) in A.2.")

#### A.2 — Summary contracts: how a cross-shot row is made

`vaft.database.summary` turns one ODS per shot into rows with a fixed definition. A **preset** is
that definition: which IDS it reads, the extractor that computes the row(s), the output columns, the
key that identifies a row and the aggregation grain. Only three presets are used in this session.

In [ ]:
PRESETS = ("shot_overview", "equilibrium_global", "core_profiles")
for name in PRESETS:
    preset = vaft.database.get_summary_preset(name)
    print(f"{name}")
    print(f"  reads IDS paths : {', '.join(preset.paths)}")
    print(f"  extractor       : {preset.extractor.__module__}.{preset.extractor.__name__}")
    print(f"  key / sort      : {preset.key_columns} / {preset.sort_columns}")
    print(f"  replace groups  : {preset.replace_groups}   ({len(preset.columns)} columns)")
    print(f"  columns         : {', '.join(preset.columns[:14])}{' ...' if len(preset.columns) > 14 else ''}")

#### A.3 — The preset's physics chain

Before copying `global_quantities`, the `equilibrium_global` extractor recomputes the global
quantities from the reconstructed profiles with VAFT's own physics functions, so every shot gets one
definition regardless of what the reconstruction code stored:

| summary column | updater (`vaft.omas`) | formula (`vaft.formula.equilibrium`) | definition |
|---|---|---|---|
| `energy_mhd_J` | `update_equilibrium_stored_energy` | `thermal_energy_from_p_V` | $W = \tfrac32 \int p\,dV$ |
| `beta_tor` | `update_equilibrium_global_quantities_beta_li` | `beta_toroidal_from_p_B0` | $\beta_t = 2\mu_0 \langle p\rangle_V / B_0^2$ |
| `beta_pol` | 〃 | `beta_poloidal_from_pressure_integral` | $\beta_p = 4\int p\,dV / (\mu_0 R_0 I_p^2)$ |
| `beta_normal` | 〃 | `beta_normal_from_beta_tor` | $\beta_N = 100\,\beta_t\, a B_0 / I_p[\mathrm{MA}]$ |
| `li_3` | 〃 | `li_3_from_Bp2_volume_integral` | $l_i(3) = 2\int B_p^2\,dV / (\mu_0^2 I_p^2 R_0)$ |
| `normalized_plasma_current` | the extractor | `normalized_plasma_current` | $I_N = I_p[\mathrm{MA}] / (a B_0)$ |
| `q_95`, `q_min` | EFIT, `update_equilibrium_global_quantities_q_min` | — | flux-surface $q$ at $\psi_N = 0.95$; its minimum |
| `volume_m3`, `area_m2`, shape | `update_equilibrium_global_quantities_volume`, `…_area`, `update_equilibrium_boundary` | — | LCFS geometry |
| `virial_*` | `compute_virial_equilibrium_quantities_ods` | `vaft.formula.virial` | Shafranov/Lao virial estimates of $\beta_p$, $l_i$ |

The remaining columns are formed in the extractor itself, from the same profiles and the LCFS:

$$
\langle p\rangle_V = \frac{1}{V}\int_V p\,dV,
\qquad
B_0 = \frac{F(\psi_b)}{R_0},\quad F = R B_\phi
\quad(\texttt{vacuum\_b0\_T},\ R_0 = 0.4\ \mathrm{m}),
$$

$$
\beta_p^{\,\mathrm{circ}} = \frac{2\mu_0\langle p\rangle_V}{B_{pa}^2},\quad
B_{pa} = \frac{\mu_0 I_p}{L_\mathrm{pol}}
\quad(\texttt{beta\_pol\_circumference}),
\qquad
\beta_{B^2} = \frac{2\mu_0\int p\,dV}{\int B^2\,dV},\quad
\beta_N^{B^2} = 100\,\beta_{B^2}\,\frac{aB_0}{I_p[\mathrm{MA}]}
\quad(\texttt{beta\_volume\_B2},\ \texttt{beta\_normal\_B2}),
$$

$$
R_\mathrm{geo} = \frac{R_\mathrm{max}+R_\mathrm{min}}{2},\quad
a = \frac{R_\mathrm{max}-R_\mathrm{min}}{2},\quad
A = \frac{R_\mathrm{geo}}{a},\quad
\kappa = \frac{Z_\mathrm{max}-Z_\mathrm{min}}{2a},\quad
\delta_{u,l} = \frac{R_\mathrm{geo}-R(Z_{\mathrm{max},\mathrm{min}})}{a},\quad
\delta = \frac{\delta_u+\delta_l}{2}.
$$

$R_0$ is `equilibrium.vacuum_toroidal_field.r0` (= 0.4 m on VEST), cross-checked against `tf.r0`. $B_0$
is rebuilt from $F$ at the boundary because the stored `b0` drifts within a shot while $F(\psi_b)$, the
vacuum $RB_\phi$, does not. In $\int B^2\,dV$ the toroidal field is $F(\psi)/R$ inside the outline, not
the vacuum field. An updater that fails leaves its columns NaN instead of stopping the shot. Part B
recomputes these numbers by hand.

In [ ]:
SAMPLE_SHOTS = (39915, 41524, 41672, 42962, 48224)
samples = {shot: vaft.omas.sample_ods(shot) for shot in SAMPLE_SHOTS}
tables = {preset: vaft.database.summary(odss=samples, preset=preset) for preset in PRESETS}
# Lab mode builds the same tables from the database instead:
# tables = {preset: vaft.database.summary(shot_range=(41000, 43000), preset=preset) for preset in PRESETS}

for preset, table in tables.items():
    print(f"{preset:20s} {len(table):3d} rows from {table['shot'].nunique()} shots")
    for failure in table.attrs["skipped"]:
        print(f"    skipped {failure['shot']} {preset}: {failure['reason']}")

Three presets, three grains: `shot_overview` gives one row per shot, `equilibrium_global` one per
EFIT slice, `core_profiles` one per kinetic-profile slice. 48224 is in `skipped` for
`shot_overview` because its sample has no plasma current. Always print `skipped`: a summary that
silently lost a third of its shots looks exactly like one that did not.

### Part B — Physical workflow: deriving the MHD parameters

The global parameters are not measured. They are **integrals over a reconstructed equilibrium**,
which itself is an inference from magnetics (and, for kinetic reconstructions, from pressure
points). The provenance diagram says which quantities are measured, reconstructed, assumed and
derived; the second diagram lists the derived global MHD quantities and the formula behind each.

In [ ]:
display(vaft.diagram.plasma_state_provenance())
display(vaft.diagram.derived_mhd_parameters(references=True))

#### B.1 — Recompute one summary row from its integrals

Take one slice of the 42962 sample (a `statistical_891` reconstruction), integrate its pressure
over the flux-surface volume and its poloidal field over the plasma cross-section, and evaluate the
`vaft.formula` functions directly. $\int B_p^2\,dV$ is done here by a plain grid quadrature inside the
LCFS, $B_p = |\nabla\psi| / (2\pi R)$ with $\psi$ in Wb (COCOS 11); the extractor uses flux-surface line
integrals, so the two agree to the accuracy of the grid.

In [ ]:
from matplotlib.path import Path as Outline
from vaft.formula import equilibrium as eq_formula

SHOT, SLICE = 42962, 3
ods = samples[SHOT]  # the summary worked on a copy; this is the caller's ODS
vaft.omas.update_equilibrium_boundary(ods)
vaft.omas.update_equilibrium_profiles_1d_geometry(ods, time_slice=[SLICE])
slice_ = ods["equilibrium.time_slice"][SLICE]

pressure = np.asarray(slice_["profiles_1d.pressure"])
volume = np.asarray(slice_["profiles_1d.volume"])
ip = float(slice_["global_quantities.ip"])
r0 = float(ods["equilibrium.vacuum_toroidal_field.r0"])
b0 = float(np.atleast_1d(ods["equilibrium.vacuum_toroidal_field.b0"])[SLICE])
a = float(slice_["boundary.minor_radius"])
pressure_integral = np.trapezoid(pressure, volume)  # int p dV [J]

grid = slice_["profiles_2d.0"]
R, Z = np.asarray(grid["grid.dim1"]), np.asarray(grid["grid.dim2"])
RR, ZZ = np.meshgrid(R, Z, indexing="ij")
dpsi_dR, dpsi_dZ = np.gradient(np.asarray(grid["psi"]), R, Z)
bp = np.hypot(dpsi_dR, dpsi_dZ) / (2 * np.pi * RR)
inside = Outline(np.column_stack([slice_["boundary.outline.r"], slice_["boundary.outline.z"]])).contains_points(
    np.column_stack([RR.ravel(), ZZ.ravel()])).reshape(RR.shape)
bp2_volume = np.sum((bp ** 2 * 2 * np.pi * RR)[inside]) * (R[1] - R[0]) * (Z[1] - Z[0])

MU0 = 4e-7 * np.pi
ip_ma = abs(ip) / 1e6
# the definitions written out, independently of vaft.formula
beta_t_hand = 2 * MU0 * (pressure_integral / volume[-1]) / b0 ** 2
by_hand = {
    "energy_mhd_J": 1.5 * pressure_integral,
    "beta_tor": beta_t_hand,
    "beta_pol": 4 * pressure_integral / (MU0 * r0 * ip ** 2),
    "beta_normal": 100 * beta_t_hand * a * abs(b0) / ip_ma,
    "li_3": 2 * bp2_volume / (MU0 ** 2 * ip ** 2 * r0),
    "normalized_plasma_current": ip_ma / (a * abs(b0)),
}
# the same quantities through the vaft.formula functions the summary updaters call
beta_t = eq_formula.beta_toroidal_from_p_B0(pressure_integral / volume[-1], b0)
by_formula = {
    "energy_mhd_J": eq_formula.thermal_energy_from_p_V(pressure_integral / volume[-1], volume[-1]),
    "beta_tor": beta_t,
    "beta_pol": eq_formula.beta_poloidal_from_pressure_integral(pressure_integral, r0, ip),
    "beta_normal": eq_formula.beta_normal_from_beta_tor(beta_t, a, b0, ip),
    "li_3": eq_formula.li_3_from_Bp2_volume_integral(bp2_volume, ip, r0),
    "normalized_plasma_current": eq_formula.normalized_plasma_current(abs(ip), r0, a, abs(b0)),
}
# select the summary row by its slice index, never by position in the table
row = tables["equilibrium_global"].query("shot == @SHOT and eq_index == @SLICE").squeeze()
comparison = pd.DataFrame({"by hand": by_hand, "vaft.formula": by_formula,
                           "summary": row[list(by_hand)].astype(float)})
comparison["relative difference"] = comparison["by hand"] / comparison["summary"] - 1
print(f"shot {SHOT}, t = {row['time_s']:.3f} s, R0 = {r0} m, B0 = {b0:.4f} T, Ip = {ip / 1e3:.1f} kA, a = {a:.3f} m")
display(comparison)

**Reading the recomputation.** $W$, $\beta_t$, $\beta_p$ and $\beta_N$ are the same numbers: they share one
pressure integral and one set of normalisations. $l_i(3)$ agrees to the accuracy of the grid
quadrature. Three consequences matter for every plane in Part E:

- $\beta_p$ and $l_i$ are normalised with $R_0$ and $I_p^2$; $\beta_t$, $\beta_N$ and $I_N$ with $B_0$ at $R_0$.
  A table that uses another radius or another current is a *different quantity* under the same name
  (Part D finds one).
- From magnetics alone, the reconstruction constrains $\beta_p + l_i/2$ much better than $\beta_p$ and
  $l_i$ separately; their scatter is a joint inference, not two measured axes.
- $q_{95}$ is a flux-surface quantity of the reconstruction; $q^*$, $q_\mathrm{cyl}$ and $I_N$ are
  algebraic in $I_p$, $a$, $\kappa$, $R$ and $B$. They are related but not interchangeable.

#### B.2 — What sets $l_i$ and $q$

$l_i$ measures how centrally the current is enclosed; $q$ is set by the current profile and the
shape. The three schematics below are the reduced (cylindrical, Miller-shape) pictures behind the
axes of Part E.

In a straight cylinder of radius $a$ and length $2\pi R$, with $B_\theta(r)$ the field of the
enclosed current,

$$
q(r) = \frac{r B_z}{R B_\theta(r)},
\qquad
l_i^{\,\mathrm{cyl}} = \frac{2\int_0^a B_\theta^2\, r\,dr}{a^2 B_\theta(a)^2},
\qquad
q(a) = \frac{2\pi a^2 B_z}{\mu_0 R I_p}.
$$

A shaped plasma has no single $q(a)$; three algebraic substitutes are in use, with $I_p$ in A:

$$
q^*_\mathrm{F} = \frac{2\pi a^2 \kappa B_0}{\mu_0 R_0 I_p}\ \text{(Freidberg)},
\qquad
q^*_\mathrm{M} = q_\mathrm{cyl}^{ST} = \frac{\pi a^2 (1+\kappa^2) B_{T0}}{\mu_0 R_0 I_p}
= \frac{2.5\,a^2(1+\kappa^2)B_T}{R\,I_p[\mathrm{MA}]}\ \text{(Menard, Sykes)},
\qquad
q_\mathrm{cyl} = \frac{5\,a^2 \kappa_a B_T}{R\,I_p[\mathrm{MA}]}.
$$

With $I_N = I_p[\mathrm{MA}]/(aB_0)$ the first one is fixed by the shape alone once multiplied by
$I_N$: $\;q^*_\mathrm{F}\, I_N = 2\pi\kappa/(\mu_0 10^6 A) = 5\kappa/A$. The two internal inductances
of this session differ only in their normalising field:

$$
l_i(3) = \frac{2\int B_p^2\,dV}{\mu_0^2 I_p^2 R_0},
\qquad
l_i(1) = \frac{\langle B_p^2\rangle_V}{B_{pa}^2}
= l_i(3)\,\frac{L_\mathrm{pol}^2 R_0}{2V}.
$$

In [ ]:
display(vaft.diagram.current_profile_shapes())
display(vaft.diagram.q_profile_landmarks())
display(vaft.diagram.shaping_family())

### Part C — MHD stability limits: models, sources and uses

A stability *limit* on an operational-space plane is one of several kinds of statement: an exact
or reduced-model criterion, an empirical boundary fitted to another machine, an experimental
record, or a solver result. The taxonomy below sorts the reduced MHD stability diagnostics by
physical problem and by that logical status.

In [ ]:
vaft.diagram.stability_diagnostic_taxonomy()

#### C.1 — The catalogue: every boundary used in Part E, with its source

`vaft.formula.boundaries` registers each boundary with its quantity, its form, its kind
(`limit`, `stability_reference`, `experimental_envelope`, …), its basis (empirical, ideal-MHD
analytic or numerical), its published source down to the equation, and the machine class it was
derived for. The table is read from the registry, not typed here.

In [ ]:
LIMITS = [  # (boundary key, plane in Part E, vaft.diagram schematic)
    ("freidberg_2008_kink_qstar", "qstar_in", "internal_external_kink"),
    ("menard_2004_qstar_min", "qstar_cyl_in", "internal_external_kink"),
    ("iter_1991_q95_min", "q95_li", "q_profile_landmarks"),
    ("low_q", "li_qa_wesson", "li_qa"),
    ("wesson_1989_jet_li_qpsi_lower", "li_qa_wesson", "li_qa"),
    ("cheng_1987_li_qa_lower", "li_qa_cheng", "li_qa"),
    ("troyon", "beta_n_li, troyon", "troyon"),
    ("strait_1994_beta_n_li", "beta_n_li1", "troyon"),
    ("greenwald_hugill", "hugill", "hugill"),
    ("murakami_hugill", "hugill, hugill_st", "hugill"),
    ("sykes_2000_st_hugill", "hugill_st", "hugill_st"),
    ("greenwald_hugill_st", "hugill_st", "hugill_st"),
]
catalogue = []
for key, plane, schematic in LIMITS:
    boundary = fb.get_boundary(key)
    source = boundary.sources[0]
    catalogue.append({
        "boundary": key, "family": boundary.family, "kind": boundary.kind, "basis": boundary.basis,
        "hardness": boundary.hardness, "quantity": f"{boundary.target.name} [{boundary.target.unit}]",
        "plane": plane, "source": f"{source.citation}; {source.equation}"[:110],
        "derived for": boundary.applicability.machine_class, "vaft.diagram": schematic,
    })
catalogue = pd.DataFrame(catalogue)
with pd.option_context("display.max_colwidth", 110, "display.width", 250):
    display(catalogue)

#### C.2 — Current limits: kink, $q^*$ and $q_{95}$

An external kink needs the displacement to reach the boundary; for $m/n$ it is driven when the edge
$q$ falls just below $m/n$. The *Kruskal–Shafranov* condition $q_a > 1$ ($m=1$) is the cylindrical
heuristic,

$$
q_a = \frac{2\pi a^2 B_0}{\mu_0 R_0 I_p} > 1 \qquad (m = 1),
$$

and the registered current limits differ in **which $q$** they use:

| boundary | basis | inequality | quantity on the axis |
|---|---|---|---|
| **Freidberg (2008) Eq. 13.162** | ideal-MHD analytic, conventional $A$, $1<\kappa<2$ | $q^*_\mathrm{F} \geq \dfrac{1+\kappa}{2}$ | $q^*_\mathrm{F} = \dfrac{2\pi a^2\kappa B_0}{\mu_0 R_0 I_p}$ (Eq. 13.160) |
| **Menard et al. (2004)** | ideal-MHD numerical (NSTX), spherical tokamaks | $q^*_\mathrm{M} \geq 1$ | $q^*_\mathrm{M} = \dfrac{\epsilon(1+\kappa^2)\pi a B_{T0}}{\mu_0 I_p}$ |
| **ITER Physics Basis (Post 1991)** | empirical design guideline | $q_{95} \geq 2.1$ (3.0 baseline) | flux-surface $q$ at $\psi_N = 0.95$ |
| **Greenwald (1988) `low_q`** | empirical | $q_\psi > 2$ | flux-surface $q$ at the boundary |

On the $(I_N, q^*)$ planes of Part E the Freidberg line is $q^*_\mathrm{F} = (1+\kappa)/2$ at one
reference $\kappa$, and the equivalent current limit is $I_p \leq 4\pi a^2 \kappa B_0/\big(\mu_0 R_0 (1+\kappa)\big)$.
Use: Freidberg is the reference current limit of an elongated tokamak, Menard the one declared for a
spherical tokamak, the ITER value a disruption-avoidance margin on $q_{95}$, not an MHD threshold.

In [ ]:
vaft.diagram.internal_external_kink()

#### C.3 — The $l_i$–$q$ domain

- **Wesson et al. (1989, JET)** — an *empirical* operating boundary from JET's 1985–88 discharges on
  $(q_\psi, l_i(3))$: rotating MHD modes during the current rise above it, kink and double-tearing
  disruptions below it. Use: a reference for how far from a large tokamak's envelope a state lies.
- **Cheng, Furth and Boozer (1987)** — *theory*: the ideal and resistive kink-stable domain of a
  pressureless straight cylinder without a wall, on $(q(a), l_i)$ of that cylinder. Use: the
  textbook picture of why too-peaked and too-broad currents are both unstable.

They are different axes from each other and from $q_{95}$; neither may receive points of another $q$.

Both curves are digitized from the published figures; there is no closed form. Their axes are

$$
\text{Wesson:}\quad \Big(q_\psi,\; l_i(3) = \frac{2\int B_\theta^2\,d\tau}{\mu_0^2 R I_p^2}\Big),\ R\ \text{the plasma major radius};
\qquad
\text{Cheng:}\quad \Big(q(a) = \frac{aB_z}{RB_\theta(a)},\; l_i^{\,\mathrm{cyl}}\Big)\ \text{of the cylinder (B.2)}.
$$

Wesson's $l_i$ uses the plasma's own $R$, the DD's $l_i(3)$ the reference $R_0$; since $l_i(3)\propto 1/R$
the two agree only where $R_\mathrm{geo} = R_0$. The registered evidence ranges are
$2 \leq q_\psi \leq 10$ (Wesson, JET 1985–88) and $2 \leq q(a) \leq 7.75$ (Cheng, $m, n \leq 20$).

In [ ]:
display(vaft.diagram.li_qa())
display(vaft.diagram.li_qa(reference="cheng_1987"))

#### C.4 — Pressure limits: Troyon and $l_i$ scaling

- **Troyon et al. (1984)** — ideal-MHD numerical: the maximum $\beta$ found for external kink and
  ballooning without a wall,

  $$
  \beta_\mathrm{max}[\%] = C_T\,\frac{I_p[\mathrm{MA}]}{a[\mathrm{m}]\,B_T[\mathrm{T}]},
  \qquad
  C_T = 2.2\,\mu_0\cdot 10^6 \approx 2.76\ \%\,\mathrm{m\,T/MA},
  \qquad
  \beta_N \equiv \beta[\%]\,\frac{aB_T}{I_p[\mathrm{MA}]} \lesssim C_T,
  $$

  with Troyon's $\beta = 2\mu_0\int p\,dV / \int B^2\,dV$ (close to $\beta_t$ at low $\beta$). Kind
  `stability_reference`: drawn without a forbidden side. On the Troyon plane it is the line
  $\beta_T = C_T I_N$ through the origin. The dashed lines are experimental records (DIII-D,
  START/PEGASUS, NSTX), not limits.
- **Strait (1994)** — empirical envelope of DIII-D and JET,

  $$
  \beta_N \leq 4\,l_i(1),
  \qquad
  l_i(1) = \frac{\langle B_p^2\rangle_V}{B_{pa}^2},\quad B_{pa} = \frac{\mu_0 I_p}{L_\mathrm{pol}},
  \qquad 1 < l_i(1) < 2 .
  $$

  It belongs on an $l_i(1)$ axis only; on an $l_i(3)$ axis the same line would move by the factor
  $L_\mathrm{pol}^2R_0/(2V)$ of each state (B.2).

In [ ]:
vaft.diagram.troyon()

#### C.5 — Density limits: Greenwald, Murakami and the Hugill diagram

The Hugill diagram plots $1/q$ against the Murakami parameter, with $\bar n_e$ the line-averaged
density in $10^{19}\,\mathrm{m^{-3}}$:

$$
x = \frac{\bar n_e R}{B_T}\ \ [10^{19}\,\mathrm{m^{-2}\,T^{-1}}],
\qquad
y = \frac{1}{q_\mathrm{cyl}},\ \ q_\mathrm{cyl} = \frac{5a^2\kappa_a B_T}{R\,I_p[\mathrm{MA}]}
\qquad\text{or}\qquad
y = \frac{1}{q^{ST}_\mathrm{cyl}},\ \ q^{ST}_\mathrm{cyl} = \frac{2.5\,a^2(1+\kappa^2) B_T}{R\,I_p[\mathrm{MA}]}.
$$

- **Murakami et al. (1976)** — empirical $\bar n_e R/B_T \lesssim 1$: a vertical line $x = 1$.
- **Greenwald et al. (1988)** — empirical $\bar n_e \leq n_G$, $n_G[10^{20}\,\mathrm{m^{-3}}] = I_p[\mathrm{MA}]/(\pi a^2)$.
  $a$, $R$ and $B_T$ cancel on the plane, leaving lines through the origin whose slope is set by the shape:

  $$
  x = \frac{50\,\kappa_a}{\pi}\,\frac{1}{q_\mathrm{cyl}}\ \ \text{(conventional)},
  \qquad
  x = \frac{25\,(1+\kappa^2)}{\pi}\,\frac{1}{q^{ST}_\mathrm{cyl}}\ \ \text{(spherical tokamak)}.
  $$

- **Sykes et al. (2000, MAST)** — the Hugill diagram re-drawn for a spherical tokamak, with the Hugill
  density $n_H = I_p[\mathrm{MA}]/(\pi a^2\kappa)$, i.e. $x = 25(1+\kappa^2)/(\pi\kappa)\,/\,q^{ST}_\mathrm{cyl}$;
  the native form for VEST.

All three are empirical; none is an MHD stability criterion, and the density used must be a
line average comparable to the source's.

In [ ]:
display(vaft.diagram.hugill())
display(vaft.diagram.hugill_st())

#### C.6 — Solver-derived classification

Part E also uses quantities that no closed formula gives; each is a number from a solver run on one
reconstructed equilibrium (Session 05), and each has a sign convention:

| criterion | definition | unstable when |
|---|---|---|
| DCON ideal energy, per $n$ | $W_t = W_p + W_v$, the least-stable eigenvalue of the plasma plus vacuum energy $\delta W[\boldsymbol\xi]$ | $W_t < 0$ |
| tearing index at $q = m/n$ (RDCON, STRIDE) | $\Delta' = \left[\dfrac{1}{\psi_1}\dfrac{d\psi_1}{dr}\right]_{r_s^-}^{r_s^+}$, the jump in the outer solution's logarithmic slope | $\Delta' > 0$ (classical drive) |
| Mercier (ideal interchange) | $D_I = E + F + H - \tfrac14$ | $D_I > 0$ |
| resistive interchange | $D_R = E + F + H^2 = D_I + (H - \tfrac12)^2$ | $D_R > 0$ |
| high-$n$ ballooning | $C_A$, evaluated only where $D_I \leq 0$ | $C_A < 0$ |

$E$, $F$, $H$ are the Glasser–Greene–Johnson (1975) surface quantities. Because $D_R \geq D_I$, a
surface can be Mercier-stable and resistive-interchange-unstable at once. None of these is a boundary
on an operating-space plane, and none of their magnitudes is a growth rate.

In [ ]:
display(vaft.diagram.dcon_rdcon_stability())
display(vaft.diagram.delta_prime())
display(vaft.diagram.interchange_criteria())

### Part D — Reading the summary tables

#### D.1 — Operational history, shot by shot

The archived history is one row per shot from the legacy pipeline-3 sheet (the source of Yun et al.,
PPCF 2025, Fig. 6). It is a **historical record, not a `shot_overview` result**: compare its columns
with the current preset contract before reading it.

In [ ]:
overview_preset = vaft.database.get_summary_preset("shot_overview")
ARCHIVED_AS = {  # current shot_overview column -> archived column holding a comparable quantity
    "plasma_onset_time_s": "plasma_onset_time  [ms, H-alpha 5 % threshold]",
    "pulse_duration_s": "pulse_duration  [ms, H-alpha 5 % threshold]",
    "max_ip_kA": "max_plasma_current  [kA, median-filtered maximum]",
    "mean_b_t_T": "toroidal_field  [T, at R = 0.4 m]",
}
contract = pd.DataFrame({"shot_overview column": [c for c in overview_preset.columns if c != "shot"]})
contract["archived history"] = contract["shot_overview column"].map(ARCHIVED_AS).fillna("— not recorded")
display(contract)
print("archived-only columns:", [c for c in history.columns if c.startswith(("has_", "efit_", "chease_"))])

In [ ]:
PICKUP_KA = 2.0  # shot_class refuses a current pulse below 2 kA as pickup (vaft.omas.shot_class)
history["current_pulse"] = history["max_plasma_current"] >= PICKUP_KA
tier_a_shots = sorted(state["shot"].unique())

figure, axes = plt.subplots(3, 1, figsize=(9, 6.8), sharex=True)
pulse = history[history["current_pulse"]]
axes[0].plot(history["shot"], history["max_plasma_current"], ".", ms=2, color="0.6", label="all rows")
axes[0].plot(pulse["shot"], pulse["max_plasma_current"], ".", ms=2, label=f"max $I_p$ >= {PICKUP_KA:g} kA")
axes[0].set_ylabel("max $I_p$ [kA]")
axes[1].plot(pulse["shot"], pulse["pulse_duration"], ".", ms=2)
axes[1].set_ylabel("H-alpha duration [ms]")
axes[2].plot(pulse["shot"], pulse["toroidal_field"], ".", ms=2)
axes[2].set_ylabel("$B_T$(0.4 m) [T]")
axes[2].set_xlabel("shot number")
axes[2].ticklabel_format(axis="x", style="plain", useOffset=False)  # shot numbers as they are, no 1e4 offset
for ax in axes:
    for shot in tier_a_shots:
        ax.axvline(shot, color="tab:red", lw=0.3, alpha=0.4)
axes[0].legend(loc="upper left", fontsize=8)
axes[0].set_title("Archived VEST operational history (red lines: the 32 Tier A shots)")
plt.tight_layout()
plt.show()

print(f"{len(history)} shots {history['shot'].min()}–{history['shot'].max()}; "
      f"{int(history['current_pulse'].sum())} with max I_p >= {PICKUP_KA:g} kA; "
      f"{int(history['has_efit_ods'].sum())} with a legacy EFIT ODS")

**Reading the history.**

- The horizontal axis is the **shot number**, not time. Shot numbers advance with every attempt,
  including vacuum and conditioning shots, and campaign breaks leave no gap. The archive stores no
  pulse date, so a calendar axis needs `pulse_time_begin` from the current `shot_overview` preset
  (D.2, and lab mode in the Integrated Analysis).
- The archive has **no shot classification**. A row with near-zero current is a failed breakdown, a
  vacuum shot or a missing signal; the current `shot_class` (Plasma / BD failure / Vacuum) separates
  those with H-alpha and barometry evidence that this table never recorded. The 2 kA cut above
  borrows only the pickup threshold of that logic; it is not a classification.
- Legacy EFIT availability is a statement about the archive, not about the plasma.
- The 32 Tier A shots sit in two narrow bands of shot numbers (39906–40331 and 42893–43002). Everything from D.3 on describes that
  band, not the whole history.

#### D.2 — The three presets on the sample shots

The tables built in Part A, read column by column.

In [ ]:
display(tables["shot_overview"])
figure, axes = vaft.plot.plot_parameter_history(
    tables["shot_overview"], y="max_ip_kA", x="date", secondary_x="shot")
axes.set_title("shot_overview: the same rows on a calendar axis (top: shot number)")
plt.show()

**How `shot_overview` decides.** `vaft.omas.plasma_timing` finds the plasma window from the slow
H-alpha filterscope (fallbacks: the fast filterscope, then the principal $I_p$ pulse) and records the
`plasma_onset_source` it used; it never assumes a window. Inside the window it takes the peak $I_p$ and
the mean vacuum field. `shot_class` is **Plasma** for a current pulse that is a discharge (or an
optical window when $I_p$ is unusable), **BD failure** when there is no pulse but the gauge or the
light responded, and **Vacuum** otherwise.

48224 is not missing by accident: its sample carries no plasma current, every timing needs $I_p$ as
a cross-check, and the summary says so in `attrs["skipped"]` instead of producing an empty row.
Always print `skipped` — a summary that silently lost a third of its shots looks exactly like one
that did not.

In [ ]:
slices = tables["equilibrium_global"]
print("slices per shot:", slices.groupby("shot").size().to_dict())
display(slices[["shot", "eq_index", "time_s", "ip_kA", "q_95", "li_3", "beta_pol", "beta_normal",
                "normalized_plasma_current", "elongation", "vacuum_r0_m"]].head(8).round(4))

`equilibrium_global` is one row per **EFIT time slice** keyed by `(shot, eq_index, time_s)`. Before
copying `global_quantities` it runs the boundary, $q_\mathrm{min}$, volume, stored-energy, $\beta$/$l_i$
and virial updaters, and an updater that fails leaves its columns NaN instead of stopping the shot.
`normalized_plasma_current` and `beta_normal` use the vacuum field at the fixed reference radius
`vacuum_r0_m` = 0.4 m.

The packaged sample equilibria are **not** the Tier A population. They show how a row is built; check
their $\beta_N$ against Part E before using them as a population (#386). D.3 onwards uses the validated
Tier A states instead.

In [ ]:
profiles_row = tables["core_profiles"]
display(profiles_row[["shot", "time_s", "ip_kA", "b_t_T", "ne_line_1e19_m3", "te_mean_eV", "p_loss_MW",
                      "tau_e_s", "tau_e_ipb89_s", "tau_e_h98y2_s", "tau_e_nstx2006l_s", "p_ohm_MW",
                      "tau_e_net_s"]].round(5).T)

`core_profiles` needs a `core_profiles` IDS **and** an equilibrium; of the five sample shots only
48224 has a Thomson-based profile, so one row comes back and the others are dropped (an extractor
that returns no row is not a failure, and is therefore not in `skipped`). The modelled
`p_loss_MW`/`tau_e_s` columns and the measured-loop `p_ohm_MW`/`tau_e_net_s` columns are kept side by
side; here the measured loop voltage is absent, so the second set is NaN rather than invented.

In [ ]:
preset = vaft.database.get_summary_preset("equilibrium_global")
written = vaft.database.export_summary(
    tables["equilibrium_global"], OUTPUT_DIR / "equilibrium_global_samples.csv",
    mode="upsert", key_columns=preset.key_columns, replace_groups=preset.replace_groups)
print(f"wrote {len(written)} rows to equilibrium_global_samples.csv in the output directory "
      f"(upsert on {preset.key_columns}, replacing whole shots)")

#### D.3 — The validated population: Tier A

150 states from 32 shots, selected as EFIT `statistical_891` slices labelled **good** or
**admissible** by the equilibrium-quality criteria (`efit_quality`). The confinement table adds geometry,
$\beta_N$ and the Ohmic power balance for the 133 magnetics-only states, and its own `selected`
flag — a *confinement* selection ($I_p$, $dW/dt$, $dI_p/dt$ rules), which is not the same thing as
equilibrium quality. These tables were built by the same physics chain as Part A, on the campaign
FileDB, and frozen.

In [ ]:
magnetics = state[state["efit_lineage"] == "magnetics"]
per_shot = magnetics.groupby("shot").size()
print(f"magnetics states: {len(magnetics)} from {per_shot.size} shots; "
      f"slices per shot min/median/max = {per_shot.min()}/{per_shot.median():.0f}/{per_shot.max()}")
print(pd.crosstab(state["efit_lineage"], state["efit_quality"], margins=True))

joined = magnetics.merge(
    confinement[KEY + ["selected", "r_geo_m", "a_m", "kappa", "kappa_area", "b_t_T", "i_p_A", "ip_efit_A",
                       "li_3", "beta_normal", "n_e_line_avg_m3", "w_mhd_J", "tau_e_th_s"]],
    on=KEY, how="outer", validate="one_to_one", indicator=True)
print("\nstate x confinement join:", joined["_merge"].value_counts().to_dict())
print("\nequilibrium quality vs confinement selection (magnetics states):")
print(pd.crosstab(joined["efit_quality"], joined["selected"].map({True: "selected", False: "not selected"})))

missing = joined[["q95", "li", "betap", "wmhd_j", "kappa", "beta_normal", "n_e_line_avg_m3", "tau_e_th_s"]].isna().mean()
print("\nmissing fraction:", missing.round(2).to_dict())
# DD poloidal beta, 4 int(p dV) / (mu0 R0 Ip^2) with W_MHD = 1.5 int(p dV), on the li_3 normalisation (R0 = 0.4 m)
MU0 = 4e-7 * np.pi
joined["beta_p_dd"] = (8 / 3) * joined["w_mhd_J"] / (MU0 * 0.4 * joined["ip_efit_A"] ** 2)
li_ratio = joined["li"] / joined["li_3"]
bp_ratio = joined["betap"] / joined["beta_p_dd"]
print(f"stored / recomputed on the same states: li {li_ratio.min():.2f}-{li_ratio.max():.2f}, "
      f"beta_p {bp_ratio.min():.2f}-{bp_ratio.max():.2f}; their quotient {float((li_ratio / bp_ratio).min()):.3f}-{float((li_ratio / bp_ratio).max()):.3f}")
# beta_pol_circumference / beta_pol is exactly L_pol^2 R0 / (2V), computed on the samples' own boundaries
samples_eq = tables["equilibrium_global"]
geometric = (samples_eq["beta_pol_circumference"] / samples_eq["beta_pol"]).dropna()
print(f"L_pol^2 R0 / (2V) on the packaged samples: {geometric.min():.2f}-{geometric.max():.2f}")
quotient = (li_ratio / bp_ratio).dropna()
if quotient.between(0.98, 1.02).all():
    print("  -> li and betap share one factor of that size: li is l_i(1) and betap the boundary-averaged beta_p,")
    print("     both normalised with B_pa = mu0 Ip / L_pol. The DD axes below use li_3 and beta_p_dd; l_i(1) is kept for Strait.")
else:
    print(f"  -> the two ratios do not share one factor ({(~quotient.between(0.98, 1.02)).sum()} states differ by > 2 %)")

In [ ]:
# one representative state per shot: the slice of maximum measured current (a modelling choice)
representative = joined.loc[joined.groupby("shot")["ip_measured_a"].idxmax()].sort_values("shot")
figure, axes = vaft.plot.plot_parameter_history(representative, y=["q95", "li_3", "beta_p_dd"], x="shot")
figure.suptitle("Tier A, one state per shot (max measured I_p)")
plt.show()

figure, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, column, label in zip(axes, ("q95", "li_3", "beta_p_dd"), ("$q_{95}$", "$l_i(3)$", r"$\beta_p$")):
    for frame, name in ((joined, f"all slices (N={len(joined)})"), (representative, f"one per shot (N={len(representative)})")):
        values = np.sort(frame[column].dropna().to_numpy())
        ax.step(values, np.arange(1, values.size + 1) / values.size, where="post", label=name)
    ax.set_xlabel(label)
axes[0].set_ylabel("ECDF")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

**Reading the population.**

- Slices of one shot are not independent samples of the machine. The two ECDFs differ where a few
  long shots contribute many slices; say which population a statistic describes.
- $q_{95}$, $l_i$, $\beta_p$ and $W_\mathrm{MHD}$ are **reconstructed** EFIT quantities, not measurements.
  From magnetics alone, EFIT constrains a combination close to $\beta_p + l_i/2$ best; separating
  $\beta_p$ from $l_i$ relies on shaping and on the assumed profile basis. Treat their scatter as a
  joint inference, not as two independent measured axes.
- The state table's `li` and `betap` are not the DD $l_i(3)$ and $\beta_p$. They are
  $l_i(1) = \langle B_p^2\rangle_V / B_{pa}^2$ and the boundary-averaged $\beta_p = 2\mu_0\langle p\rangle_V / B_{pa}^2$,
  both with $B_{pa} = \mu_0 I_p / L_\mathrm{pol}$, so both differ from the DD forms by the same factor
  $L_\mathrm{pol}^2 R_0 / (2V)$. The cell prints that factor computed on the packaged samples' own boundaries
  (`beta_pol_circumference / beta_pol`) and the two Tier A ratios, which agree with each other to about 1 %
  (the residual is the recomputed $l_i(3)$). Same names, different
  quantities: the DD planes use `li_3` and `beta_p_dd`, and the stored `li` is used only where
  $l_i(1)$ is the quantity (Strait, E.1).
- `efit_quality` (is this a trustworthy reconstruction?) and `selected` (is this slice usable for an
  Ohmic power balance?) answer different questions; neither removes rows from the plots below.
  Missing densities (about half the states) are a Thomson-coverage fact, not a plasma fact.
- The observed $q_{95}$ range is an **envelope** of where VEST was operated in this campaign subset,
  not a limit.

#### D.4 — Confinement, briefly

The confinement table is a DB5-like Ohmic power balance from the measured loop voltage, radiation not
subtracted:

$$
P_\mathrm{net} = P_\mathrm{OH} - \frac{dW}{dt},
\qquad
\tau_E = \frac{W_\mathrm{MHD}}{P_\mathrm{net}},
\qquad
H = \frac{\tau_E^\mathrm{obs}}{\tau_E^\mathrm{scal}} .
$$

A confinement scaling is a power law in engineering variables, fitted to a multi-machine database,

$$
\tau_E^\mathrm{scal}[\mathrm{s}] = C\; I_p^{\alpha_I}\, B_T^{\alpha_B}\, \bar n_{19}^{\alpha_n}\, P^{\alpha_P}\,
R^{\alpha_R}\, \epsilon^{\alpha_\epsilon}\, \kappa^{\alpha_\kappa}\, M^{\alpha_M},
$$

with $I_p$ in MA, $B_T$ in T at $R$, $\bar n_{19}$ the line-averaged density in $10^{19}\,\mathrm{m^{-3}}$,
$P$ the loss power in MW, $R$ the geometric major radius in m, $\epsilon = a/R$, $\kappa$ the area
elongation and $M$ the effective ion mass in amu. The three used here, as registered in
`vaft.formula` (`confinement_time_from_engineering_parameters`):

$$
\begin{aligned}
\text{ITER89P (L-mode, global } W\text{)}:&\quad
\tau_E = 0.038\, I_p^{0.85} B_T^{0.20} \bar n_{19}^{0.10} P^{-0.50} R^{1.50} \epsilon^{0.30} \kappa^{0.50} M^{0.50},\\
\text{IPB98(y,2) (ELMy H-mode, thermal } W\text{)}:&\quad
\tau_E = 0.0562\, I_p^{0.93} B_T^{0.15} \bar n_{19}^{0.41} P^{-0.69} R^{1.97} \epsilon^{0.58} \kappa^{0.78} M^{0.19},\\
\text{NSTX 2006 L-mode (Kaye et al., global } W\text{)}:&\quad
\tau_E = 0.141\, I_p^{1.01} B_T^{0.70} \bar n_{19}^{0.07} P^{-0.37}.
\end{aligned}
$$

The NSTX fit has no size, shape or mass term: it was regressed on one machine. An $H$ factor is
the conventional one only on the scaling's own energy basis (`vaft.formula.confinement_scaling_basis`):
a thermal scaling against $\tau_{E,\mathrm{th}} = W_\mathrm{th}/P$, a global one against
$\tau_{E} = W/P$. None of the three was fitted on a 0.4 m spherical tokamak in Ohmic operation, so an
$H$ factor here measures distance from another machine class, not confinement quality.

In [ ]:
confinement_table = public.load_vest_tier_a_confinement(None)  # every state; `selected` marks the primary selection
SCALINGS = ("ITER89P", "H98y2", "NSTX2006L")
for scaling in SCALINGS:
    h = public.h_factor(confinement_table, scaling, thermal_as_global={"VEST"})
    confinement_table[f"H_{scaling}"] = h
    confinement_table[f"tau_e_{scaling}_s"] = public.predict_confinement_time(confinement_table, scaling)
    basis = vaft.formula.confinement_scaling_basis(scaling)
    print(f"{scaling:10s} H median (selected) = {h[confinement_table['selected']].median():.2f}   "
          f"finite rows {int(h.notna().sum())}/{len(h)}   energy basis: {basis.energy_basis} "
          f"(compared with {h.attrs['energy_basis']})")
print(f"Thomson line-averaged density present: {int(confinement_table['n_e_line_avg_m3'].notna().sum())}/{len(confinement_table)} states")

chosen = confinement_table[confinement_table["selected"]]
figure, axes = plt.subplots(1, len(SCALINGS), figsize=(12, 4), sharex=True, sharey=True)
observed = chosen["tau_e_th_s"] * 1e3
finite = [chosen[f"tau_e_{s}_s"] * 1e3 for s in SCALINGS] + [observed]
low = 0.5 * float(np.nanmin([series[series > 0].min() for series in finite]))
high = 2.0 * float(np.nanmax([series.max() for series in finite]))
diagonal = np.array([low, high])
for ax, scaling in zip(axes, SCALINGS):
    predicted = chosen[f"tau_e_{scaling}_s"] * 1e3
    both = predicted.notna() & observed.notna()
    ax.loglog(predicted[both], observed[both], "o", ms=4, alpha=0.75, color="tab:blue")
    ax.plot(diagonal, diagonal, "k-", lw=1.0, label="H = 1")
    ax.plot(diagonal, 2 * diagonal, "k:", lw=0.8, label="H = 2, 1/2")
    ax.plot(diagonal, 0.5 * diagonal, "k:", lw=0.8)
    ax.set_xlim(low, high)
    ax.set_ylim(low, high)
    ax.set_aspect("equal")
    ax.set_title(f"{scaling}: median H = {chosen[f'H_{scaling}'].median():.2f} (N = {int(both.sum())})", fontsize=9)
    ax.set_xlabel(r"$\tau_E^\mathrm{scal}$ [ms]")
axes[0].set_ylabel(r"$\tau_E^\mathrm{obs} = W_\mathrm{MHD}/P_\mathrm{net}$ [ms]")
axes[0].legend(fontsize=7, loc="upper left")
figure.suptitle("Confinement scalings against the selected Tier A states with a Thomson density", fontsize=10)
plt.tight_layout()
plt.show()

**Reading the scalings.** The two conventional-tokamak scalings put most VEST states between $H = 1/2$
and $H = 1$. The NSTX L-mode fit predicts several times the observed $\tau_E$: it was fitted on NSTX
alone and has no size term that would scale it down to a 0.4 m machine. The printed medians are the
numbers to quote; the offset from $H = 1$ is a machine-class difference, not a confinement mode.

A scaling needs the line-averaged density, so only the states with a Thomson profile at the EFIT
time get a prediction. `thermal_as_global={"VEST"}` uses the thermal $\tau_E$ where a scaling is
defined on a global stored energy — acceptable for an Ohmic plasma without fast ions, and recorded
in `attrs`. The kinetic reconstructions (17 electron-kinetic states) are sparse in time, so kinetic
quantities are matched to the magnetics state at exactly the same EFIT time, never to the nearest
available slice.

### Part E — Operating space with the stability limits

#### E.1 — The planes

The population for the planes below is the 133 magnetics-only Tier A states: the electron-kinetic
states carry no separate geometry here. Columns are renamed to the projection's quantity names and
every unit is declared; `vaft.plot.operational_space` then draws the boundaries of Part C only where
the axis identity and unit match, records what it left out in `ax.vaft_overlay.omitted`, and states
for each drawn boundary whether its declared domain covers the plotted states in
`ax.vaft_applicability`.

| plane | $x$ | $y$ | boundaries drawn (Part C) |
|---|---|---|---|
| `qstar_in` | $I_N = I_p[\mathrm{MA}]/(aB_T)$ | $q^*_\mathrm{F}$ | Freidberg $q^*_\mathrm{F} = (1+\kappa)/2$ |
| `qstar_cyl_in` | $I_N$ | $q^*_\mathrm{M} = q^{ST}_\mathrm{cyl}$ | Menard $q^*_\mathrm{M} = 1$ |
| `q95_li` | $q_{95}$ | $l_i(3)$ | ITER $q_{95} = 2.1$ |
| `li_qa_wesson`, `li_qa_cheng` | $q_\psi$; $q(a)$ | $l_i(3)$; $l_i^{\,\mathrm{cyl}}$ | Wesson, Greenwald `low_q`; Cheng (reference only) |
| `beta_n_li` | $l_i(3)$ | $\beta_N$ | Troyon $\beta_N = C_T$ |
| `beta_n_li1` | $l_i(1)$ | $\beta_N$ | Troyon, Strait $\beta_N = 4\,l_i(1)$ |
| `troyon` | $I_N$ | $\beta_T[\%] = \beta_N I_N$ | Troyon $\beta_T = C_T I_N$ |
| `hugill`, `hugill_st` | $\bar n_e R/B_T$ | $1/q_\mathrm{cyl}$; $1/q^{ST}_\mathrm{cyl}$ | Murakami, Greenwald; Sykes |

$B_T$ in $I_N$ and $\beta_N$ is the vacuum field at $R_\mathrm{geo}$ for the coordinate helpers, so the
stored $\beta_N(R_0)$ is converted once, $\beta_N(R_\mathrm{geo}) = \beta_N(R_0)\,R_\mathrm{geo}/R_0$
(Physical Context, item 3).

In [ ]:
mag = joined[joined["_merge"] == "both"].copy()
# the EFIT current: beta_N, li_3 and the shape come from the same equilibrium. (i_p_A, the measured
# current of the power balance, differs from it by 0.69-1.12 on these states.)
ip_ma = mag["ip_efit_A"].to_numpy() / 1e6
r_geo, a, kappa, b_geo = (mag[c].to_numpy() for c in ("r_geo_m", "a_m", "kappa", "b_t_T"))
R0_REFERENCE = 0.4  # m: VAFT's stored beta_N uses the vacuum field there
beta_conversion = r_geo / R0_REFERENCE  # B(R0)/B(R_geo) for a 1/R vacuum field

opspace = pd.DataFrame({
    "shot": mag["shot"].to_numpy(), "time_us": mag["time_us"].to_numpy(),
    "efit_lineage": mag["efit_lineage"].to_numpy(), "efit_quality": mag["efit_quality"].to_numpy(),
    "time_efit_s": mag["time_efit_s"].to_numpy(),
    "normalized_current": ip_ma / (a * b_geo),
    "kink_safety_factor_elliptic": fb.kink_coordinates(a, r_geo, b_geo, kappa, ip_ma),
    "kink_safety_factor_cylindrical": fb.cylindrical_kink_coordinates(a, r_geo, b_geo, kappa, ip_ma),
    "internal_inductance_li3": mag["li_3"].to_numpy(),
    "internal_inductance_li1": mag["li"].to_numpy(),  # the state table's li is l_i(1) (D.3)
    "edge_safety_factor_95": mag["q95"].to_numpy(),
    "normalized_beta": mag["beta_normal"].to_numpy() * beta_conversion,
    "elongation": kappa, "area_elongation": mag["kappa_area"].to_numpy(),
})
opspace["toroidal_beta"] = opspace["normalized_beta"] * opspace["normalized_current"]
UNITS = {"normalized_current": "MA m^-1 T^-1", "normalized_beta": "% m T/MA", "toroidal_beta": "%",
         "murakami_parameter": "1e19 m^-2 T^-1"}
MACHINE_CLASS = "spherical tokamak"  # VEST: R/a ~ 1.4
opspace.attrs["machine_class"] = MACHINE_CLASS

print(f"{len(opspace)} states. Field convention: B_T at R_geo (vacuum, |b0 r0|/R_geo) for I_N, q* and Hugill;")
print(f"beta_N converted from B(R0 = {R0_REFERENCE} m) to B(R_geo): factor R_geo/R0 = "
      f"{beta_conversion.min():.3f}–{beta_conversion.max():.3f} (median {np.median(beta_conversion):.3f}).")
print(f"aspect ratio R/a: {np.median(r_geo / a):.2f} (median); elongation median {np.median(kappa):.2f}")

In [ ]:
KAPPA_REFERENCE = float(np.median(kappa))
figure, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, projection, label in ((axes[0], "qstar_in", "Freidberg $q^*$"), (axes[1], "qstar_cyl_in", "Menard (ST) $q^*$")):
    operational_space_population(
        opspace, projection, color="efit_quality", units=UNITS, boundary_inputs={"elongation": KAPPA_REFERENCE},
        boundary_style="inline", ax=ax, title=f"{label} against $I_N$")
plt.tight_layout()
plt.show()
for ax in axes:
    print(ax.get_title()[:24], "omitted:", ax.vaft_overlay.omitted, "| applicability:",
          {key: value for key, value in ax.vaft_applicability.items()})
shape = opspace["kink_safety_factor_elliptic"] * opspace["normalized_current"]
print(f"q* x I_N = 5 kappa / A: {shape.min():.2f}-{shape.max():.2f} -> on this plane the states lie along shape curves")
print(f"Freidberg boundary at the reference elongation {KAPPA_REFERENCE:.2f}; Menard's q* >= 1 has no free input")

In [ ]:
# The DCON qlim of the atlas is evaluated at psi_N = psilim, not on the last closed flux surface:
print("DCON psilim (full edge):", sorted(atlas_n["dcon_full_256_psilim"].round(4).unique()))
# so no q_psi(LCFS) column exists here; li_qa_pair draws both references without data points.
figure, axes = li_qa_pair(opspace[["internal_inductance_li3"]])
plt.show()

figure, ax = operational_space_population(opspace, "q95_li", color="efit_quality", units=UNITS,
                                          title="$l_i(3)$ against $q_{95}$: the data, with no boundary from another q")
plt.show()
print("omitted   :", ax.vaft_overlay.omitted)

The Wesson $l_i(3)$–$q_\psi$ survey and the Cheng–Furth–Boozer cylindrical $l_i$–$q(a)$ domain are drawn
without VEST points: the atlas' DCON `qlim` is evaluated at $\psi_N = 0.994$, not at the last closed
flux surface, so no $q_\psi$ column exists. The VEST states appear on the $q_{95}$–$l_i$ plane, whose only
registered boundary is the ITER design guideline $q_{95} \geq 2.1$ — far below every VEST state.

In [ ]:
figure, ax = operational_space_population(opspace, "beta_n_li", color="efit_quality", units=UNITS,
                                          title=r"$\beta_N$ against $l_i(3)$")
plt.show()
drawn = [curve.key for curve in ax.vaft_overlay.curves]
print("drawn     :", drawn)
print("Strait 4 l_i(1) on this l_i(3) axis:", "strait_1994_beta_n_li" in drawn)
print("applicable:", ax.vaft_applicability)

In [ ]:
figure, ax = operational_space_population(opspace, "beta_n_li1", color="efit_quality", units=UNITS,
                                          title=r"$\beta_N$ against $l_i(1)$ (Strait's quantity)")
plt.show()
drawn = [curve.key for curve in ax.vaft_overlay.curves]
print("drawn     :", drawn)
print(f"l_i(1) range: {opspace['internal_inductance_li1'].min():.2f}-{opspace['internal_inductance_li1'].max():.2f} "
      "(Strait's evidence range is 1-2)")
print("applicable:", ax.vaft_applicability)

In [ ]:
figure, ax = operational_space_population(opspace, "troyon", color="efit_quality", units=UNITS,
                                          title=r"$\beta_T$ against $I_N$ (both at $R_\mathrm{geo}$)")
plt.show()
ratio = opspace["toroidal_beta"] / opspace["normalized_current"]
print("beta_T here is derived, beta_T = beta_N * I_N: max |beta_T/I_N - beta_N| =",
      f"{float(np.abs(ratio - opspace['normalized_beta']).max()):.1e}")
print("so every point's slope from the origin is its beta_N; the plane adds no independent beta measurement")
print("applicable:", {key: value[0] for key, value in ax.vaft_applicability.items()})

In [ ]:
with_density = mag["n_e_line_avg_m3"].notna().to_numpy()
print(confinement["n_e_definition"].iloc[0])
print(f"states with a line-averaged density: {int(with_density.sum())}/{len(mag)}")
dense = mag[with_density]
x, y = fb.hugill_coordinates(dense["n_e_line_avg_m3"].to_numpy() / 1e19, dense["r_geo_m"].to_numpy(),
                             dense["b_t_T"].to_numpy(), dense["a_m"].to_numpy(),
                             dense["kappa_area"].to_numpy(), dense["ip_efit_A"].to_numpy() / 1e6)
x_st, y_st = fb.hugill_coordinates_st(dense["n_e_line_avg_m3"].to_numpy() / 1e19, dense["r_geo_m"].to_numpy(),
                                      dense["b_t_T"].to_numpy(), dense["a_m"].to_numpy(),
                                      dense["kappa"].to_numpy(), dense["ip_efit_A"].to_numpy() / 1e6)
density_plane = pd.DataFrame({"murakami_parameter": x, "inverse_cylindrical_q": y, "inverse_cylindrical_q_st": y_st,
                              "elongation": dense["kappa"].to_numpy(), "area_elongation": dense["kappa_area"].to_numpy(),
                              "efit_quality": dense["efit_quality"].to_numpy()})
density_plane.attrs["machine_class"] = MACHINE_CLASS

figure, axes = plt.subplots(1, 2, figsize=(11, 4.3))
operational_space_population(density_plane, "hugill_st", color="efit_quality", units=UNITS, ax=axes[0])
operational_space_population(density_plane, "hugill", color="efit_quality", units=UNITS, ax=axes[1])
plt.tight_layout()
plt.show()
for ax in axes:
    print(ax.get_title()[:40], "->", {key: value[0] for key, value in ax.vaft_applicability.items()})

**Reading the planes.**

- Read the printed applicability before reading a boundary: their declared calibration classes and
  ranges (conventional tokamaks, JET, DIII-D, MAST/NSTX surveys) need not include a 0.4 m spherical
  tokamak at $I_N \approx 2$. Unless a boundary is SUPPORTED, it is a reference, not a VEST limit.
- Two current limits, two $q^*$: Freidberg's elongated-tokamak line is OUTSIDE its calibration for VEST,
  Menard's $q^* \geq 1$ is declared for spherical tokamaks but has no tested range (UNASSESSED). Neither
  is a VEST limit until such a range covers VEST.
- The Freidberg line is drawn at one reference elongation; each state has its own $\kappa$. Because
  $q^* I_N = 5\kappa/A$, the $q^*$–$I_N$ plane is close to a one-parameter family set by the shape:
  moving along a curve is changing current at fixed shape, not exploring a second dimension.
- The Troyon plane only re-expresses $\beta_N$; its information is the same as the $\beta_N$ axis of
  the $\beta_N$–$l_i$ plane. Strait's $\beta_N \leq 4 l_i$ is defined with $l_i(1)$: it is drawn on the
  $l_i(1)$ plane, outside its 1 < $l_i$ < 2 evidence range for most VEST states, and never on $l_i(3)$.
- On the Hugill planes only states with a Thomson density are plotted; the spherical-tokamak form
  (Sykes 2000) is the native one for VEST, the conventional one is shown for comparison.

#### E.2 — Joining the stability atlas, state by state

The atlas is per state and $n$. For the per-state view each layer is reduced explicitly:

- **ideal (DCON), per edge treatment**: *unstable* if any $n$ is `robust_unstable` (the sign of
  $W_t<0$ agrees at two resolutions); *stable* only if all six $n$ are `robust_stable`; *unresolved*
  otherwise. Full-edge and truncated-edge energies are separate verdicts.
- **classical tearing drive, per solver (RDCON, STRIDE)**: *QA-supported drive* if a $\Delta'>0$
  surface is two-resolution consistent at $n=1$ or $2$; *raw drive only* if $\Delta'>0$ appears only
  without that support; *no drive found* if both $n$ were resolved without one; *not assessed* otherwise.
- **local**: Mercier $D_I>0$, resistive interchange $D_R>0$, ballooning $C_A<0$, as stored.

Written out for one state $s$ and one edge treatment, with $N = \{1,\dots,6\}$ and $S_{n}$ the rational
surfaces of $n$:

$$
\text{ideal}(s) =
\begin{cases}
\text{unstable} & \exists\, n\in N:\ \texttt{sign\_class}_n = \text{robust\_unstable}\ (W_t<0 \text{ at both resolutions}),\\
\text{stable} & \forall\, n\in N:\ \texttt{sign\_class}_n = \text{robust\_stable},\\
\text{unresolved} & \text{otherwise};
\end{cases}
$$

$$
\text{drive}(s) =
\begin{cases}
\text{QA-supported} & \exists\, n\in\{1,2\},\ k\in S_n:\ \Delta'_k > 0 \ \wedge\ \texttt{two\_resolution\_consistent}_k,\\
\text{raw only} & \exists\, n, k:\ \Delta'_k > 0,\ \text{none of them consistent},\\
\text{none found} & n = 1, 2 \text{ both resolved and } \Delta'_k \leq 0\ \forall k,\\
\text{not assessed} & \text{otherwise}.
\end{cases}
$$

In [ ]:
ideal = atlas_n.assign(
    full_unstable=atlas_n["dcon_full_sign_class"].eq("robust_unstable"),
    full_stable=atlas_n["dcon_full_sign_class"].eq("robust_stable"),
    trunc_unstable=atlas_n["dcon_trunc_sign_class"].eq("robust_unstable"),
    trunc_stable=atlas_n["dcon_trunc_sign_class"].eq("robust_stable"),
).groupby(KEY).agg(n_modes=("n_tor", "nunique"), full_unstable=("full_unstable", "any"),
                   full_stable=("full_stable", "all"), trunc_unstable=("trunc_unstable", "any"),
                   trunc_stable=("trunc_stable", "all"), mercier_unstable=("ideal_interchange_unstable", "any"),
                   resistive_interchange_unstable=("resistive_interchange_unstable", "any"),
                   ballooning_unstable=("ballooning_unstable", "any"))
for edge in ("full", "trunc"):
    ideal[f"ideal_{edge}_edge"] = np.select(
        [ideal[f"{edge}_unstable"], ideal[f"{edge}_stable"] & ideal["n_modes"].eq(6)],
        ["unstable", "stable"], default="unresolved")
ideal["unstable_n_full_edge"] = (atlas_n[atlas_n["dcon_full_sign_class"].eq("robust_unstable")]
                                 .groupby(KEY)["n_tor"].apply(list))

resistive = atlas_n[atlas_n["n_tor"].isin([1, 2])]
for solver in ("rdcon", "stride"):
    per_state = resistive.assign(
        # .eq(True): a failed run leaves the flag empty (NaN), which bool() would turn into a drive
        raw=resistive[f"{solver}_classical_tearing_drive"].eq(True),
        qa=resistive[f"{solver}_classical_tearing_drive_qa"].eq(True),
        resolved=resistive[f"{solver}_qa_status"].eq("RESOLVED"),
    ).groupby(KEY).agg(raw=("raw", "any"), qa=("qa", "any"), resolved=("resolved", "all"))
    ideal[f"{solver}_drive"] = np.select(
        [per_state["qa"], per_state["raw"], per_state["resolved"]],
        ["QA-supported drive", "raw drive only", "no drive found"], default="not assessed")
    ideal[f"{solver}_both_n_resolved"] = per_state["resolved"]  # False: n=1 or 2 failed or did not resolve

stability_state = ideal.reset_index()
report = state.merge(stability_state, on=KEY, how="outer", validate="one_to_one", indicator=True)
counts = report["_merge"].value_counts()
print("join report (state x per-state stability)")
print(f"  states in v1/state.csv           : {len(state)}")
print(f"  states in the atlas              : {len(stability_state)} (reduced from {len(atlas_n)} state x n rows)")
print(f"  matched / state only / atlas only: {counts.get('both', 0)} / {counts.get('left_only', 0)} / {counts.get('right_only', 0)}")
print(f"  duplicate keys after the join    : {int(report.duplicated(KEY).sum())}")
print(f"  matched by lineage               : {report.loc[report['_merge'] == 'both', 'efit_lineage'].value_counts().to_dict()}")

opspace = opspace.merge(stability_state, on=KEY, how="left", validate="one_to_one")
opspace.attrs["machine_class"] = MACHINE_CLASS
print(f"operating-space states with a stability row: {int(opspace['ideal_full_edge'].notna().sum())}/{len(opspace)}")

In [ ]:
labelled = report[report["_merge"] == "both"]
print("ideal, full edge  :", labelled["ideal_full_edge"].value_counts().to_dict())
print("ideal, trunc. edge:", labelled["ideal_trunc_edge"].value_counts().to_dict())
print("RDCON  n=1,2      :", labelled["rdcon_drive"].value_counts().to_dict())
print("STRIDE n=1,2      :", labelled["stride_drive"].value_counts().to_dict())
print("a verdict above rests on one n only (the other failed or did not resolve): RDCON",
      int((~labelled["rdcon_both_n_resolved"] & labelled["rdcon_drive"].ne("not assessed")).sum()),
      " STRIDE", int((~labelled["stride_both_n_resolved"] & labelled["stride_drive"].ne("not assessed")).sum()))
print("Mercier D_I > 0   :", int(labelled["mercier_unstable"].sum()),
      "  resistive interchange D_R > 0:", int(labelled["resistive_interchange_unstable"].sum()),
      "  ballooning C_A < 0:", int(labelled["ballooning_unstable"].sum()))
print("\nideal (full edge) x RDCON drive, states:")
print(pd.crosstab(labelled["ideal_full_edge"], labelled["rdcon_drive"], margins=True))
print("\nRDCON x STRIDE:")
print(pd.crosstab(labelled["rdcon_drive"], labelled["stride_drive"]))
unstable = labelled[labelled["ideal_full_edge"] == "unstable"]
display(unstable[KEY + ["efit_quality", "unstable_n_full_edge", "ideal_trunc_edge", "rdcon_drive", "stride_drive"]])

In [ ]:
IDEAL_COLORS = {"stable": "#2a9d8f", "unstable": "#d62828", "unresolved": "#9e9e9e"}
figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
for ax, projection in zip(axes, ("qstar_in", "beta_n_li")):
    operational_space_population(
        opspace, projection, color="ideal_full_edge", marker="rdcon_drive", hollow=("not assessed",),
        category_colors=IDEAL_COLORS, units=UNITS, ax=ax, marker_size=26,
        boundary_inputs={"elongation": KAPPA_REFERENCE} if projection == "qstar_in" else None,
        title=f"{projection}: colour = DCON full edge, marker = RDCON drive")
plt.tight_layout()
plt.show()

**Reading the classification.**

- The ideal verdict depends on the **edge treatment**: the full-edge energy is sensitive to the
  last surfaces kept (`psihigh`), and most full-edge instabilities disappear with the
  truncated edge. Neither is "the" answer; report both.
- A classical tearing drive ($\Delta'>0$) appears on most states when every surface counts, and on
  far fewer when only two-resolution-consistent surfaces do. That gap is a statement about numerical
  QA, not about tearing. RDCON and STRIDE do not always agree; they are kept apart. An $n$ whose run
  did not resolve can still contribute a raw drive; only the QA-supported count excludes it.
- Resistive interchange $D_R>0$ is widespread while Mercier $D_I>0$ is rare: the two local criteria
  answer different questions.
- Look at where the unstable states sit on the $q^*$–$I_N$ and $\beta_N$–$l_i$ planes before claiming
  that either plane predicts the solver verdict.

#### E.3 — Stability distributions

`vaft.process.mhd_stability.stability_atlas_populations` reduces the atlas to per-slice values per
panel — DCON $W_t$ per $n$ (full edge, mpsi 512), the largest $\Delta'$ over rational surfaces per solver
and $n$, and the extreme local criteria — without any QA cut on the values. The `qa` column of its
summary says how many of those values pass their QA. `stability_atlas_population` draws them.

In [ ]:
populations = mhd_stability.stability_atlas_populations(atlas_n, atlas_surfaces)
figure, axes = stability_atlas_population(populations)
plt.show()
display(populations.summary)
print(populations.provenance)

The sign carries the physics: $W_t<0$ is ideal-unstable at any magnitude, and $\Delta'>0$ is a
classical drive at any magnitude. Magnitudes are not growth rates, and no magnitude band
("marginal") is drawn without an independently derived threshold. The $\Delta'$ panel spans
many decades because $\Delta'$ grows large near low-shear surfaces and $q_\mathrm{min}$; the `qa` count,
not the axis range, says which values are numerically supported.

In [ ]:
# The pipeline-3 stability sheets predate this atlas. Read them only to see why they are not plotted.
LEGACY = data_path("").parents[1] / "workflow" / "automatic_pipeline_3_data_summary"
if (LEGACY / "chease_dcon_history.xlsx").is_file():
    legacy = pd.read_excel(LEGACY / "chease_dcon_history.xlsx")
    stability_sheet = pd.read_excel(LEGACY / "stability_history.xlsx")
    deltas = stability_sheet.filter(like="delta_w_n").to_numpy(dtype=float)
    legacy_beta_n = legacy["beta_normal"]
    print(f"chease_dcon_history.xlsx: {len(legacy)} rows; beta_N median {legacy_beta_n.median():.3f} "
          f"(Tier A: {confinement['beta_normal'].median():.2f})")
    print(f"stability_history.xlsx  : {len(stability_sheet)} rows; ideal labels "
          f"{stability_sheet['ideal_stability'].value_counts(dropna=False).to_dict()}")
    print(f"  delta_w_n* min {np.nanmin(deltas):.3g}; one scalar per run, no resolution check, no surface identity")
else:
    print("workflow/ is not present (installed package): the legacy sheets are not available here")

The legacy sheets classify with fixed magnitude cut-offs ("Error" above $|W| \geq 100$ or
$|\Delta'| \geq 1000$, then dropped before the join), keep one $\Delta'$ per run without its surface, and
store a $\beta_N$ far below the Tier A value for comparable states. They are the reason the atlas keeps
raw values, QA and criteria as separate columns.

## Interpretation Checkpoints

1. **The archived history has no pulse dates.** What would change in D.1 if the axis were calendar
   time, and which preset column provides it?
2. **48224 is in `skipped`.** Why is a skipped shot better than a row of NaN, and where would you look
   for the reason in lab mode?
3. **More than one slice per shot.** Which of the two ECDFs in D.3 would you compare with another
   machine's population, and why?
4. **`selected` and `efit_quality`.** Describe a state that is a good equilibrium but not a usable
   power balance, and say which flag each plot should use.
5. **No VEST point is drawn on the Wesson panel.** Which quantity is missing, and why may $q_{95}$ or DCON
   `qlim` not stand in for it?
6. **The Troyon plane.** What does it tell you that the $\beta_N$ axis of the $\beta_N$–$l_i$ plane does not?
7. **A state is ideal-unstable with the full edge and stable with the truncated edge.** What do you
   report, and what further calculation would decide between them?
8. **RDCON shows a QA-supported $\Delta'>0$.** Is the state tearing-unstable? What is missing?

## Integrated Analysis

### Rebuild the population from the database (lab mode)

In lab mode the same analysis starts from the database instead of the frozen collections: the
`shot_overview` history over a shot range with calendar dates and shot classes, and the
`equilibrium_global` slices of the same shots. Everything is read-only; nothing is written back.

In [ ]:
LAB = os.environ.get("VAFT_TUTORIAL_MODE", "offline") == "lab"
if LAB:
    # The summary skips a shot it cannot open and says why in the log (a 403 is missing credentials);
    # keep those messages visible here, or an unreachable database reads as an empty one.
    logging.getLogger("vaft.database").setLevel(logging.WARNING)
    SHOT_RANGE = (41000, 43000)
    lab_overview = vaft.database.summary(shot_range=SHOT_RANGE, preset="shot_overview")
    lab_slices = vaft.database.summary(shot_range=SHOT_RANGE, preset="equilibrium_global")
    for name, table in (("shot_overview", lab_overview), ("equilibrium_global", lab_slices)):
        print(f"{name}: {len(table)} rows, {len(table.attrs['skipped'])} shots skipped")
    if lab_overview.empty:
        raise RuntimeError("the database returned no shot_overview rows: check the logged reasons above")
    print(lab_overview["shot_class"].value_counts().to_dict())
    plasma = lab_overview[lab_overview["shot_class"] == "Plasma"]
    figure, axes = vaft.plot.plot_parameter_history(plasma, y=["max_ip_kA", "pulse_duration_s"], x="date", secondary_x="shot")
    plt.show()
else:
    print("Offline: the database population needs lab mode (VAFT_TUTORIAL_MODE=lab and HSDS read access).\n"
          "Parts D and E used the frozen Tier A collection and the packaged samples instead.")

### Optional extensions

Once a population is defined and validated, the usual next steps are: regression of one descriptor
on others **split by shot** (slices of one shot are not independent), nearest-neighbour similarity in
standardised dimensionless variables, joining event records (current quench, disruption) to the last
state before the event **by time**, and learning models on the population. VAFT provides the
primitives (`vaft.process.statistical_analysis`, `vaft.process.transients`, `vaft.process.ml`); none
of them changes what Parts A–E established about provenance, coverage and quantity identity.

## Independent Exercise

1. Use the **truncated-edge** verdict instead of the full-edge one in the E.2 overlay. Which states
   change colour, and where do they sit on the $q^*$–$I_N$ plane?
2. Replace the RDCON drive by the STRIDE drive as the marker. Count the states on which the two solvers
   disagree about a QA-supported drive.
3. Choose a different representative-state rule in D.3 (for example maximum $W_\mathrm{MHD}$) and
   compare the per-shot history.
4. Restrict the operating-space planes to `selected` states and say which conclusions change.
5. *(lab)* Rebuild `shot_overview` for another shot range and plot the `shot_class` fractions per
   calendar month.

In [ ]:
# Lab mode: VAFT_TUTORIAL_MODE=lab and HSDS read access for exercise 5.
# 1. operational_space_population(opspace, "qstar_in", color="ideal_trunc_edge", units=UNITS, ...)
# 2. pd.crosstab(labelled["rdcon_drive"], labelled["stride_drive"])

## Takeaways and Next Steps

- A population study starts from definitions: a summary preset fixes the source IDS, the extractor,
  the key and the grain of every row, and `attrs["skipped"]` says which shots it could not build.
- Archived tables are history, not presets: check their columns against the current contract
  before reading them, and keep shot number and calendar time apart.
- Lazy access (`open`) reads a few leaves of many shots; eager access (`load`) materialises one shot;
  `summary(odss=...)` runs the same extractor offline.
- **An observed envelope is not an operating limit**, and a reference boundary is drawn only on the
  quantity it was defined in, with its domain stated.
- Join tables by state key and time at their own grain, and print the join report.
- Ideal, resistive and local stability criteria, and their numerical QA, are separate layers. Not
  assessed is not stable, and $\Delta'>0$ is not a tearing verdict.

**Formula sheet** (all lengths in m, fields in T, $I_p$ in MA unless written in A):

| quantity | definition | used on |
|---|---|---|
| $W$, $\beta_t$, $\beta_p$ | $\tfrac32\int p\,dV$; $\;2\mu_0\langle p\rangle_V/B_0^2$; $\;4\int p\,dV/(\mu_0 R_0 I_p^2)$ | summary, Part B |
| $\beta_N$, $I_N$ | $100\,\beta_t\,aB_0/I_p$; $\;I_p/(aB_0)$ | Troyon, $\beta_N$–$l_i$, $q^*$–$I_N$ |
| $l_i(3)$, $l_i(1)$ | $2\int B_p^2\,dV/(\mu_0^2 I_p^2 R_0)$; $\;l_i(3)\,L_\mathrm{pol}^2R_0/(2V)$ | $\beta_N$–$l_i$, Wesson |
| $q^*_\mathrm{F}$, $q^*_\mathrm{M}$ | $2\pi a^2\kappa B_0/(\mu_0R_0I_p)$; $\;\pi a^2(1+\kappa^2)B_{T0}/(\mu_0R_0I_p)$ ($I_p$ in A) | current limits |
| current limits | $q^*_\mathrm{F} \geq (1+\kappa)/2$; $\;q^*_\mathrm{M}\geq 1$; $\;q_{95}\geq 2.1$; $\;q_\psi>2$ | C.2 |
| pressure limits | $\beta_N \lesssim 2.76$ (Troyon); $\;\beta_N \leq 4\,l_i(1)$ (Strait, $1<l_i<2$) | C.4 |
| density limits | $\bar n_e R/B_T \lesssim 1$ (Murakami); $\;\bar n_e \leq I_p/(\pi a^2)$ [$10^{20}\,\mathrm{m^{-3}}$] (Greenwald) | C.5 |
| solver criteria | $W_t<0$; $\;\Delta'>0$; $\;D_I>0$; $\;D_R>0$; $\;C_A<0$ = unstable / drive | C.6, E.2 |

**Open work this session depends on**: a dated, classified campaign `shot_overview` snapshot for the
offline course; a validated $q_\psi$(LCFS) column for the Wesson plane; inner-layer matching (RMATCH)
for a tearing verdict.

**This is the end of the course.** Session 01 opened one discharge; session 06 has put it among the
others. The next step is your own question on the database.